In [1]:
# ======================
# IMPORTS
# ======================
import os
import json
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.preprocessing import image

print("TensorFlow:", tf.__version__)

# ======================
# CONFIG
# ======================
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

BASE_DIR = os.getcwd()
DATA_DIR = os.path.join(BASE_DIR, "dataset")
MODEL_DIR = os.path.join(BASE_DIR, "model")

os.makedirs(MODEL_DIR, exist_ok=True)

print("Dataset path:", DATA_DIR)
print("Exists:", os.path.exists(DATA_DIR))

# ======================
# DATA GENERATOR
# ======================
datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    zoom_range=0.3,
    horizontal_flip=True,
    brightness_range=[0.7, 1.3],
    validation_split=0.2
)

train_gen = datagen.flow_from_directory(
    DATA_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training'
)

val_gen = datagen.flow_from_directory(
    DATA_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation'
)

print("Classes:", train_gen.class_indices)

# ======================
# SAVE LABELS
# ======================
labels_path = os.path.join(MODEL_DIR, "spinach_labels.json")

class_labels = {v: k for k, v in train_gen.class_indices.items()}

with open(labels_path, "w") as f:
    json.dump(class_labels, f, indent=4)

# ======================
# MODEL
# ======================
base_model = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(train_gen.num_classes, activation='softmax')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# ======================
# CALLBACKS
# ======================
checkpoint_path = os.path.join(MODEL_DIR, "spinach_model.keras")

callbacks_list = [
    callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    callbacks.ModelCheckpoint(filepath=checkpoint_path, monitor='val_accuracy', save_best_only=True, verbose=1)
]

# ======================
# TRAINING
# ======================
model.fit(train_gen, epochs=3, validation_data=val_gen, callbacks=callbacks_list)

# ======================
# FINE TUNE
# ======================
base_model.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.fit(train_gen, epochs=3, validation_data=val_gen, callbacks=callbacks_list)

print("Training complete!")

# ======================
# PREDICTION FUNCTION
# ======================
def predict_disease(img_path):
    model = tf.keras.models.load_model(checkpoint_path)

    with open(labels_path, "r") as f:
        labels = json.load(f)

    img = image.load_img(img_path, target_size=IMG_SIZE)
    img_array = image.img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    preds = model.predict(img_array)[0]
    class_idx = np.argmax(preds)
    confidence = float(preds[class_idx])

    return {
        "disease": labels[str(class_idx)],
        "confidence": round(confidence * 100, 2)
    }

TensorFlow: 2.21.0
Dataset path: F:\krishi nova project ml part\spinach\dataset
Exists: True
Found 2407 images belonging to 5 classes.
Found 599 images belonging to 5 classes.
Classes: {'Anthracnose(102)': 0, 'Bacterial-Spot(752)': 1, 'Downy-Mildew(240)': 2, 'Healthy-Leaf(1399)': 3, 'Pest-Damage(513)': 4}


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ efficientnetb0 (Functional)          │ (None, 7, 7, 1280)          │       4,049,571 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d             │ (None, 1280)                │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization                  │ (None, 1280)                │           5,120 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 256)                 │         327,936 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 256)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 5)                   │           1,285 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 4,383,912 (16.72 MB)

 Trainable params: 331,781 (1.27 MB)

 Non-trainable params: 4,052,131 (15.46 MB)

Epoch 1/3
76/76 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.3452 - loss: 1.7862
Epoch 1: val_accuracy improved from None to 0.46578, saving model to F:\krishi nova project ml part\spinach\model\spinach_model.keras

Epoch 1: finished saving model to F:\krishi nova project ml part\spinach\model\spinach_model.keras
76/76 ━━━━━━━━━━━━━━━━━━━━ 253s 3s/step - accuracy: 0.3565 - loss: 1.7667 - val_accuracy: 0.4658 - val_loss: 1.4355
Epoch 2/3
76/76 ━━━━━━━━━━━━━━━━━━━━ 0s 701ms/step - accuracy: 0.3821 - loss: 1.6953
Epoch 2: val_accuracy did not improve from 0.46578
76/76 ━━━━━━━━━━━━━━━━━━━━ 66s 870ms/step - accuracy: 0.3743 - loss: 1.6919 - val_accuracy: 0.4658 - val_loss: 1.4487
Epoch 3/3
76/76 ━━━━━━━━━━━━━━━━━━━━ 0s 692ms/step - accuracy: 0.3887 - loss: 1.6245
Epoch 3: val_accuracy did not improve from 0.46578
76/76 ━━━━━━━━━━━━━━━━━━━━ 65s 859ms/step - accuracy: 0.4051 - loss: 1.5566 - val_accuracy: 0.4658 - val_loss: 1.4743
Epoch 1/3
76/76 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 